<a href="https://colab.research.google.com/github/IllangasingheIMDP/Neural-Networks/blob/main/P2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define the main project path (you can rename 'RideSafe-MAS' if you prefer)
project_path = '/content/drive/MyDrive/RideSafe-MAS/P2_Historical_Flood'

# 3. Define the subdirectories based on the project tasks
directories = [
    f'{project_path}/wb_flood_events',       # For P2.1: World Bank data
    f'{project_path}/official_reports',      # For P2.4: DMC/Irrigation reports
    f'{project_path}/processed_data'         # For outputs like geojsons and csvs
]

# 4. Create the directories if they don't exist
for directory in directories:
    os.makedirs(directory, exist_ok=True)
    print(f"Ready: {directory}")

print("\nEnvironment setup complete! Your Google Drive is ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ready: /content/drive/MyDrive/RideSafe-MAS/P2_Historical_Flood/wb_flood_events
Ready: /content/drive/MyDrive/RideSafe-MAS/P2_Historical_Flood/official_reports
Ready: /content/drive/MyDrive/RideSafe-MAS/P2_Historical_Flood/processed_data

Environment setup complete! Your Google Drive is ready.


In [8]:
# Define paths based on our previous setup
project_path = '/content/drive/MyDrive/RideSafe-MAS/P2_Historical_Flood'
reports_dir = f'{project_path}/official_reports'

Water level and rainfall from dmc

In [ ]:
# import os
# import requests
# from bs4 import BeautifulSoup
# from urllib.parse import urljoin
# import re
# import urllib3

# # Suppress insecure request warnings
# urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# # --- Configuration ---
# START_DATE = "2023-01-01"
# END_DATE = "2025-12-31"

# URL = f"https://www.dmc.gov.lk/index.php?Itemid=277&fromdate={START_DATE}&lang=en&limit=0&option=com_dmcreports&report_type_id=6&search=&todate={END_DATE}&view=reports"
# BASE_URL = "https://www.dmc.gov.lk"
# DOWNLOAD_DIR = "dmc_reports_2023_2025"

# # Create a folder in Colab to store the files
# os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# print(f"Fetching page content and extracting links...")
# print(f"Target URL: {URL}")

# def download_reports():
#     # 1. Fetch the main page
#     response = requests.get(URL, verify=False)

#     if response.status_code != 200:
#         print(f"Failed to load the page. HTTP Status Code: {response.status_code}")
#         return

#     soup = BeautifulSoup(response.text, 'html.parser')

#     # 2. Locate all <a> tags that contain the text "Download"
#     download_tags = soup.find_all('a', string=lambda text: text and 'Download' in text)
#     print(f"Found {len(download_tags)} reports between {START_DATE} and {END_DATE}.\nStarting download process...\n")

#     # 3. Iterate through links, extract date/time, and download
#     for idx, tag in enumerate(download_tags):
#         href = tag.get('href')
#         if not href:
#             continue

#         download_url = urljoin(BASE_URL, href)

#         # --- NEW: Extract Date and Time from the HTML Table Row ---
#         date_str = "UnknownDate"
#         time_str = "UnknownTime"

#         # Find the parent table row (<tr>) of this specific download button
#         row = tag.find_parent('tr')
#         if row:
#             cols = row.find_all('td')
#             # The webpage table columns are: [0] Title, [1] Date, [2] Time, [3] Download
#             if len(cols) >= 3:
#                 date_str = cols[1].get_text(strip=True)
#                 # Replace ':' with '-' because colons can cause issues in Windows/Linux file paths
#                 time_str = cols[2].get_text(strip=True).replace(':', '-')

#         try:
#             # Stream the file to handle large downloads efficiently
#             file_response = requests.get(download_url, verify=False, stream=True)
#             file_response.raise_for_status()

#             # --- NEW: Extract original extension, fallback to .pdf ---
#             ext = ".pdf"
#             cd = file_response.headers.get('content-disposition')

#             if cd:
#                 fname_match = re.findall('filename=(.+)', cd)
#                 if fname_match:
#                     original_filename = fname_match[0].strip(' "').replace(';', '')
#                     if '.' in original_filename:
#                         # Grab whatever extension the server actually sent
#                         ext = "." + original_filename.split('.')[-1]

#             # Combine everything into the final filename
#             filename = f"Report_{date_str}_{time_str}{ext}"
#             file_path = os.path.join(DOWNLOAD_DIR, filename)

#             # Write file chunks to disk
#             with open(file_path, 'wb') as f:
#                 for chunk in file_response.iter_content(chunk_size=1024*1024):
#                     if chunk:
#                         f.write(chunk)

#             print(f"[{idx+1}/{len(download_tags)}] Downloaded: {filename}")

#         except Exception as e:
#             print(f"[{idx+1}/{len(download_tags)}] Failed to download link ({download_url}): {e}")

#     print("\n✅ All downloads complete! Check the folder on the left panel in Colab.")

# # Execute the function
# download_reports()

In [6]:
!pip install geopandas requests tools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00


In [9]:
import os
import requests
import zipfile
import geopandas as gpd

# Define paths
project_path = '/content/drive/MyDrive/RideSafe-MAS/P2_Historical_Flood'
processed_dir = f'{project_path}/processed_data'
temp_extract_dir = '/content/unosat_temp'

# UNOSAT Shapefile URL from your JSON payload
unosat_url = "https://unosat.org/static/unosat_filesystem/3868/FL20240603LKA_SHP.zip"
zip_path = "/content/unosat_2024.zip"

# 1. Download the zip file
print("Downloading UNOSAT Shapefile...")
response = requests.get(unosat_url, stream=True)
with open(zip_path, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)
print("✅ Download complete.")

# 2. Extract the zip file
print("Extracting files...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(temp_extract_dir)

# 3. Locate and load the shapefile into GeoPandas
# Look for the main .shp file inside the extracted folder
shp_file = None
for root, dirs, files in os.walk(temp_extract_dir):
    for file in files:
        if file.endswith(".shp"):
            shp_file = os.path.join(root, file)
            break

if shp_file:
    print(f"Found Shapefile: {shp_file}")
    print("Converting to GeoJSON...")

    # Load shapefile
    gdf = gpd.read_file(shp_file)

    # Ensure it's in standard WGS84 (EPSG:4326) coordinate system
    gdf = gdf.to_crs(epsg=4326)

    # Define output path
    output_geojson = f'{processed_dir}/unosat_2024_water_extent.geojson'

    # Save as GeoJSON
    gdf.to_file(output_geojson, driver='GeoJSON')
    print(f"✅ Successfully created and saved: {output_geojson}")
else:
    print("❌ Error: No .shp file found in the downloaded archive.")

✅ Download complete.
Extracting files...
Found Shapefile: /content/unosat_temp/FL20240603LKA_SHP/S1_20240604_WaterExtent_SouthWest_SriLanka.shp
Converting to GeoJSON...
✅ Successfully created and saved: /content/drive/MyDrive/RideSafe-MAS/P2_Historical_Flood/processed_data/unosat_2024_water_extent.geojson
